# Pandera SchemaError の出力を確認する

Pandera でバリデーションに失敗したとき、**どのカラムの・何行目の・どんな値が問題か**がエラーから分かることを、実際の出力で確認します。

本ノートブックではこのプロジェクトの `InferenceSchema` / `TrainSchema` を使います。

In [ ]:
import pandas as pd
import pandera.pandas as pa
from pandera.errors import SchemaError
from pandera.typing import Series

from f1_pit_stops.schema import InferenceSchema, TrainSchema


def show_schema_error(df: pd.DataFrame, schema: type[pa.DataFrameModel], label: str) -> None:
    """バリデーションを実行し、失敗時はエラー内容と failure_cases を表示する。"""
    print(f"=== {label} ===")
    try:
        schema.validate(df)
        print("OK: バリデーション成功")
    except SchemaError as e:
        print(type(e).__name__ + ":")
        print(e)
        print()
        print("failure_cases（行番号と問題の値）:")
        print(e.failure_cases.to_string(index=False))
    print()

## 1. 正常系（比較用）

まず問題のないデータではエラーが出ないことを確認します。

In [ ]:
ok_df = pd.DataFrame([{
    "Stint": 2,
    "Year": 2024,
    "Driver": "HAM",
    "Race": "Monaco Grand Prix",
    "TyreLife": 15.0,
    "RaceProgress": 0.4,
    "Compound": "MEDIUM",
}])

show_schema_error(ok_df, InferenceSchema, "正常な推論データ")

=== 正常な推論データ ===
OK: バリデーション成功



## 2. `Compound` に想定外の値（記事の例に相当）

42行目だけ `Compound` を `"INTER"` に差し替えます。  
`isin` チェックが失敗し、**index 42 / failure_case INTER** が `failure_cases` に出ます。

In [ ]:
rows = []
for _ in range(43):
    rows.append({
        "Stint": 2,
        "Year": 2024,
        "Driver": "HAM",
        "Race": "Monaco Grand Prix",
        "TyreLife": 15.0,
        "RaceProgress": 0.4,
        "Compound": "MEDIUM",
    })
rows[42]["Compound"] = "INTER"  # 許可されていない値

bad_compound_df = pd.DataFrame(rows)
show_schema_error(bad_compound_df, InferenceSchema, "Compound に INTER が混入")

=== Compound に INTER が混入 ===
SchemaError:
Column 'Compound' failed element-wise validator number 0: isin(['HARD', 'MEDIUM', 'SOFT', 'INTERMEDIATE', 'WET']) failure cases: INTER

failure_cases（行番号と問題の値）:
 index failure_case
    42        INTER



## 3. 数値範囲の違反（`TyreLife`）

`TyreLife` の上限は 77.0 です。100.0 を入れると、同様に行番号と値が特定できます。

In [ ]:
bad_tyre_df = pd.DataFrame([
    {
        "Stint": 1,
        "Year": 2024,
        "Driver": "VER",
        "Race": "Monaco Grand Prix",
        "TyreLife": 10.0,
        "RaceProgress": 0.2,
        "Compound": "SOFT",
    },
    {
        "Stint": 1,
        "Year": 2024,
        "Driver": "VER",
        "Race": "Monaco Grand Prix",
        "TyreLife": 100.0,  # 上限 77.0 を超える
        "RaceProgress": 0.3,
        "Compound": "SOFT",
    },
])

show_schema_error(bad_tyre_df, InferenceSchema, "TyreLife が上限超過")

=== TyreLife が上限超過 ===
SchemaError:
Column 'TyreLife' failed element-wise validator number 1: less_than_or_equal_to(77.0) failure cases: 100.0

failure_cases（行番号と問題の値）:
 index  failure_case
     1         100.0



## 4. 学習用スキーマ（`TrainSchema`）での目的変数エラー

`PitNextLap` は 0 か 1 のみ許可されます。

In [ ]:
bad_train_df = pd.DataFrame([
    {
        "id": 0,
        "Stint": 1,
        "Year": 2024,
        "Driver": "NOR",
        "Race": "Monaco Grand Prix",
        "TyreLife": 5.0,
        "RaceProgress": 0.1,
        "Compound": "HARD",
        "PitNextLap": 0,
    },
    {
        "id": 1,
        "Stint": 1,
        "Year": 2024,
        "Driver": "NOR",
        "Race": "Monaco Grand Prix",
        "TyreLife": 6.0,
        "RaceProgress": 0.2,
        "Compound": "HARD",
        "PitNextLap": 2,  # 0/1 以外
    },
])

show_schema_error(bad_train_df, TrainSchema, "PitNextLap が 0/1 以外")

=== PitNextLap が 0/1 以外 ===
SchemaError:
Column 'PitNextLap' failed element-wise validator number 0: isin([0, 1]) failure cases: 2

failure_cases（行番号と問題の値）:
 index  failure_case
     1             2



## 5. 記事の例と同じ書き方（簡易スキーマ）

説明用に、記事で挙げた `compound` / `SOFT|MEDIUM|HARD` の最小スキーマでも同じことが起きることを確認します。

In [ ]:
class DemoSchema(pa.DataFrameModel):
    compound: Series[str] = pa.Field(isin=["SOFT", "MEDIUM", "HARD"])


demo_rows = [{"compound": "MEDIUM"} for _ in range(43)]
demo_rows[42]["compound"] = "INTER"
demo_df = pd.DataFrame(demo_rows)

show_schema_error(demo_df, DemoSchema, "記事と同型の compound 例")

=== 記事と同型の compound 例 ===
SchemaError:
Column 'compound' failed element-wise validator number 0: isin(['SOFT', 'MEDIUM', 'HARD']) failure cases: INTER

failure_cases（行番号と問題の値）:
 index failure_case
    42        INTER



In [10]:
# 1.5 → int に coerce すると、エラーなく 1 になる
df = pd.DataFrame({"val": [1.5, 2.5, 3.5]})
TestSchema.validate(df)  # 成功。val は [1, 2, 3] になっている

NameError: name 'TestSchema' is not defined